# Tracker Optuna Tuning — Google Colab H100

**Repo:** `YGTKL16/tekli_obje_takibi`  
**Baseline:** `configs/c1_startup_classifier.yaml` → FinalScore = 0.7215  
**Tuner:** `scripts/tune_all_params_canary.py` (Optuna TPE)

**Çalıştırma sırası:** Hücreleri sırasıyla çalıştırın (Ctrl+F9 = tümünü çalıştır).

In [ ]:
# ── 0. AYARLAR ──────────────────────────────────────────────────────────────
import os

REPO_URL       = "https://github.com/YGTKL16/tekli_obje_takibi"
REPO_PATH      = "/content/tekli_obje_takibi"

# Google Drive'daki dosya yolları
DRIVE_ZIP      = "/content/drive/MyDrive/contest_release.zip"          # veri seti zip
DRIVE_WEIGHTS  = "/content/drive/MyDrive/sglatrack_ep0297.pth.tar"     # model ağırlıkları

# Optuna ayarları
USE_SGLA       = True      # True = PyTorch (Colab'de TRT yok)
RUN_SMOKE      = True      # Kurulumu doğrulamak için kısa test koşusu
RUN_FULL       = True      # Asıl optimizasyon
SMOKE_TRIALS   = 3
FULL_TRIALS    = 100
STUDY_NAME     = "all_params_h100_v1"
BASE_CONFIG    = "configs/c1_startup_classifier.yaml"  # başlangıç noktası

SMOKE_SEQS = [
    "dataset5/uav4",
    "dataset3/truck_night",
    "dataset3/car8",
]
# FULL_SEQS boş = tune_all_params_canary.py kendi varsayılan setini kullanır
FULL_SEQS = []

# Türetilmiş yollar — değiştirme
STUDY_DB       = os.path.join(REPO_PATH, "cache", "optuna_studies", f"{STUDY_NAME}.db")
CHECKPOINT_DIR = os.path.join(REPO_PATH, "cache", "optuna_studies")
DATA_DIR       = "/content/data"
WEIGHTS_DEST   = os.path.join(REPO_PATH, "models", "SGLATrack", "checkpoints", "sglatrack_ep0297.pth.tar")

print(f"Repo    : {REPO_PATH}")
print(f"Backend : {'SGLA/PyTorch' if USE_SGLA else 'TensorRT'}")
print(f"Study DB: {STUDY_DB}")

In [ ]:
# ── 1. GOOGLE DRIVE BAĞ ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. VERİ SETİ ZİP ÇIKAR ─────────────────────────────────────────────────
import zipfile

if os.path.exists(DRIVE_ZIP):
    print(f"Zip bulundu: {DRIVE_ZIP}")
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f"Çıkartıldı → {DATA_DIR}")
else:
    raise FileNotFoundError(
        f"{DRIVE_ZIP} bulunamadı.\n"
        "contest_release.zip dosyasını Google Drive ana dizinine yükleyin."
    )

In [ ]:
# ── 3. REPO KLONLA / GÜNCELLE ───────────────────────────────────────────────
import subprocess, sys, re

def run(cmd: str, check: bool = True) -> str:
    """Shell komutu çalıştır, son 4000 karakter çıktıyı bas."""
    proc = subprocess.run(cmd, shell=True, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = proc.stdout or ""
    print(out[-4000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"Komut başarısız ({proc.returncode}): {cmd}")
    return out


# Bastırılacak gürültülü satır kalıpları (TF/CUDA/timm/ffmpeg uyarıları)
_NOISE_RE = re.compile(
    r"FutureWarning"
    r"|timm\.models\.(helpers|layers|registry)"
    r"|oneDNN custom operations"
    r"|cuFFT factory|cuDNN factory|cuBLAS factory"
    r"|computation placer already registered"
    r"|absl::InitializeLog"
    r"|TensorFlow binary is optimized"
    r"|AVX2 AVX512|rebuild TensorFlow"
    r"|warnings\.warn"
    r"|E0000 |W0000 |I tensorflow"
    r"|mov,mp4,m4a|moov atom not found"
    r"|Using an existing study"
)


def run_stream(cmd: str) -> int:
    """Shell komutu çalıştır, çıktıyı satır satır canlı yayınla.

    Uzun süren komutlar (Optuna, cmake) için kullanın.
    TF/CUDA/timm/ffmpeg gürültü uyarıları otomatik olarak bastırılır.
    """
    env = os.environ.copy()
    env.update({
        "TF_CPP_MIN_LOG_LEVEL": "3",
        "TF_ENABLE_ONEDNN_OPTS": "0",
        "PYTHONWARNINGS": "ignore",
    })
    proc = subprocess.Popen(
        cmd, shell=True, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env=env,
    )
    assert proc.stdout is not None
    for line in iter(proc.stdout.readline, ""):
        if not _NOISE_RE.search(line):
            print(line, end="", flush=True)
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        print(f"[uyarı] çıkış kodu={rc}")
    return rc

if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    print("Repo klonlanıyor...")
    run(f"rm -rf {REPO_PATH}")
    run(f"git clone {REPO_URL} {REPO_PATH}")
else:
    print("Repo mevcut, güncelleniyor...")
    run(f"cd {REPO_PATH} && git pull")

# Veri setini repoya sembolik link ile bağla
target = os.path.join(REPO_PATH, "data", "contest_release")
os.makedirs(os.path.join(REPO_PATH, "data"), exist_ok=True)
if not os.path.exists(target):
    try:
        os.symlink(DATA_DIR, target)
        print(f"Symlink oluşturuldu: {target} → {DATA_DIR}")
    except FileExistsError:
        print("Symlink zaten mevcut.")


In [ ]:
# ── 4. MODEL AĞIRLIKLARINI KOPYALA ─────────────────────────────────────────
import shutil

os.makedirs(os.path.dirname(WEIGHTS_DEST), exist_ok=True)

if os.path.exists(WEIGHTS_DEST):
    print("Ağırlık dosyası zaten mevcut.")
elif os.path.exists(DRIVE_WEIGHTS):
    print(f"Kopyalanıyor: {DRIVE_WEIGHTS} → {WEIGHTS_DEST}")
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_DEST)
    print("Kopyalama tamamlandı.")
else:
    raise FileNotFoundError(
        f"{DRIVE_WEIGHTS} bulunamadı.\n"
        "sglatrack_ep0297.pth.tar dosyasını Google Drive ana dizinine yükleyin."
    )

In [ ]:
# ── 5. SİSTEM BAĞIMLILIKLARI ─────────────────────────────────────────────────
run("apt-get remove -y pybind11-dev 2>/dev/null || true", check=False)
run("apt-get update -qq && apt-get install -y -q "
    "libeigen3-dev libopencv-dev python3-dev libturbojpeg build-essential cmake")

In [ ]:
# ── 6. PYTHON BAĞIMLILIKLARI ─────────────────────────────────────────────────
run(f"{sys.executable} -m pip install -q "
    "numpy opencv-python PyYAML torch torchvision timm easydict "
    "optuna tqdm gdown onnx pybind11 pytest")

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 7. SGLATrack KURULUMU ────────────────────────────────────────────────────
os.chdir(REPO_PATH)

run("bash scripts/download_sglatrack.sh", check=False)
run("bash scripts/setup_sglatrack.sh", check=False)

# torch._six uyumluluk yamaları
loader = os.path.join(REPO_PATH, "models/SGLATrack/lib/train/data/loader.py")
if os.path.exists(loader):
    run(f"sed -i 's/from torch._six import string_classes/string_classes = str/g' {loader}")
    run(f"sed -i 's/from torch._six import int_classes/int_classes = int/g' {loader}")
    print("Uyumluluk yamaları uygulandı.")

In [ ]:
# ── 8. C++ tracker_cpp DERLEME ───────────────────────────────────────────────
os.chdir(REPO_PATH)

pybind11_cmake = subprocess.check_output(
    [sys.executable, "-c", "import pybind11; print(pybind11.get_cmake_dir())"]
).decode().strip()

BUILD_DIR = "build_colab"
run(f"rm -rf {BUILD_DIR}")
run(f"cmake -B {BUILD_DIR} -DCMAKE_BUILD_TYPE=Release "
    f"-DPYTHON_EXECUTABLE={sys.executable} "
    f"-DPython3_EXECUTABLE={sys.executable} "
    f"-Dpybind11_DIR={pybind11_cmake}")
run(f"cmake --build {BUILD_DIR} -j$(nproc) --target tracker_cpp")

# .so dosyasını repo köküne kopyala
run(f'find {BUILD_DIR} -name "tracker_cpp*.so" -exec cp {{}} {REPO_PATH}/tracker_cpp.so \\;')

# Doğrula
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "python"))
import tracker_cpp  # type: ignore
print("tracker_cpp başarıyla import edildi.")

In [ ]:
# ── 9. SMOKE RUN (Kurulum Doğrulama) ───────────────────────────────────────
import shlex

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def seq_args(seqs):
    return " ".join(f"--seq {s}" for s in seqs)

backend_arg = "--use-sgla" if USE_SGLA else ""

if RUN_SMOKE:
    smoke_db = os.path.join(CHECKPOINT_DIR, f"{STUDY_NAME}_smoke.db")
    cmd = (
        f"PYTHONPATH={REPO_PATH}:{REPO_PATH}/python "
        f"{sys.executable} scripts/tune_all_params_canary.py "
        f"{backend_arg} "
        f"--base-config {BASE_CONFIG} "
        f"--n-trials {SMOKE_TRIALS} "
        f"--study-name {STUDY_NAME}_smoke "
        f"--study-db {smoke_db} "
        f"--checkpoint-dir {CHECKPOINT_DIR} "
        f"{seq_args(SMOKE_SEQS)}"
    )
    print("=== SMOKE RUN BAŞLIYOR ===")
    run_stream(cmd)
    print("=== SMOKE RUN BİTTİ ===")
else:
    print("RUN_SMOKE=False, atlandı.")

In [ ]:
# ── 10. FULL OPTUNA RUN ──────────────────────────────────────────────────────
if RUN_FULL:
    cmd = (
        f"PYTHONPATH={REPO_PATH}:{REPO_PATH}/python "
        f"{sys.executable} scripts/tune_all_params_canary.py "
        f"{backend_arg} "
        f"--base-config {BASE_CONFIG} "
        f"--n-trials {FULL_TRIALS} "
        f"--study-name {STUDY_NAME} "
        f"--study-db {STUDY_DB} "
        f"--checkpoint-dir {CHECKPOINT_DIR} "
        f"{seq_args(FULL_SEQS)}"
    )
    print("=== FULL RUN BAŞLIYOR ===")
    run_stream(cmd)
    print("=== FULL RUN BİTTİ ===")
else:
    print("RUN_FULL=False, atlandı.")

In [ ]:
# ── 11. SONUÇLAR: Optuna Görselleştirme ─────────────────────────────────────
import optuna
import optuna.visualization as vis

if not os.path.exists(STUDY_DB):
    print(f"Veritabanı bulunamadı: {STUDY_DB}")
    print("Full run tamamlanmadan bu hücreyi çalıştırmayın.")
else:
    study = optuna.load_study(
        study_name=STUDY_NAME,
        storage=f"sqlite:///{STUDY_DB}"
    )
    n = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    print(f"Tamamlanan deneme sayısı: {n}")

    if n > 0:
        best = study.best_trial
        print(f"\nEn iyi değer (FinalScore): {-best.value:.4f}")
        print("En iyi parametreler:")
        for k, v in best.params.items():
            print(f"  {k}: {v}")

        vis.plot_optimization_history(study).show()

        try:
            vis.plot_param_importances(study).show()
        except Exception as e:
            print(f"Parametre önem grafiği çizilemedi: {e}")

        try:
            vis.plot_parallel_coordinate(study).show()
        except Exception as e:
            print(f"Paralel koordinat grafiği çizilemedi: {e}")

In [ ]:
# ── 12. EN İYİ CONFIG'İ DRIVE'A KAYDET ─────────────────────────────────────
import glob

# checkpoint-dir'de kaydedilen en iyi YAML dosyalarını listele
yamls = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, f"{STUDY_NAME}*.yaml")))
if yamls:
    print("Kaydedilen config dosyaları:")
    for y in yamls:
        print(" ", y)

    # Drive'a kopyala
    drive_out = "/content/drive/MyDrive/"
    for y in yamls:
        dest = os.path.join(drive_out, os.path.basename(y))
        shutil.copy(y, dest)
        print(f"Drive'a kopyalandı: {dest}")
else:
    print("Henüz kaydedilmiş config yok.")